# Extraction des bases de données intermédiaires

Quelles sont les données disponibles sur les émeutes en France ? 
Nous avons cherché à obtenir des données spatiales et temporelles sur des évènements catégorisés comme des émeutes par le passé. 

## Base de données du GDELT Project
Nous nous sommes intéressés dans un premier temps aux données du GDELT Project, qui analyse des articles de presses publiés partout dans le monde sur des sujets très variés et les regroupe dans une même base de données en open data. Elle contient en particulier des évènements concernant les manifestations et plus précisément encore les émeutes.

Face à la grande taille de cette base de données, nous avons utilisé le service Google Cloud BigQuery pour effectuer un filtre sur les émeutes en France via une requête SQL, et nous sommes pour cela passés par une API et un compte de services. Pour éviter de scanner un trop gros volume de données et ainsi dépasser la limite d'utilisation gratuite de Google BigQuery de 1 To de données scannées, nous avons utilisé le partitionnement de la base sur la date, pour extraire seulement les évènements émeutes en France postérieurs à 2023, en filtrant sur les colonnes qui nous paraissaient utiles.

En effet, la Base du GDELT project contient 61 colonnes qui ne sont pas toutes utiles pour notre approche. Dans cette base, un évènements est décrit comme une interaction entre au plus deux acteurs, avec des informations sur l'interaction et sur chacun des deux acteurs. La documentation est disponible [ici](https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf). 

Nous avons donc choisi pour notre analyse de conserver :

informations générales :
- GLOBALEVENTID : identifiant de l'évènement, pour d'éventuelles jointures ultérieures (pour rajouter les colonnes écartées par exemple)
- SOURCEURL : l'URL de l'article source sur lequel se base le codage machine de l'évènement (pour pouvoir comprendre la source du codage de l'évènement et vérifier qualitativement la qualité du codage de certains évènements)
- SQLDATE : la date

informations sur les acteurs 1 (resp. 2 en remplaçant 1 par 2 dans les noms de variables) :
- Actor1CountryCode : le code CAMEO pour le pays de l'acteur 1
- Actor1Name : le nom de l'acteur 1
- Actor1Type1Code, Actor1Type2Code, Actor1Type3Code : des champs qui précisent la catégorie de l'acteur 1
- Actor1Geo_FeatureID : un code GNIS de la localisation de l'acteur lorsque la précision le permet, une chaine de caractère sinon (peut permettre des jointures spatiales)

informations sur la source et son environnement médiatique
- NumArticles : le nb de sources qui parlent du meme évènement dans un ontervalle de temps resseré (pour évaluer l'importance de l'évènement)
- AvgTone : positif si les articles qui parlent de cet évènement en parle en moyenne positivement (mesure subjective)

informations sur la localisation de l'évènement :
- ActionGeo_Type : la précision du géocodage, 4 désigne une ville hors US, 5 un découpage administratif de niveau 1 (région) hors US
- ActionGeo_CountryCode : l'état dans lequel se déroule l'évènement
- ActionGeo_FullName : une chaine de caractère dénommant le lieu
- ActionGeo_ADM1Code : présente un code FIPS10-4 (en France, un équivalent de la région)
- ActionGeo_ADM2Code : présente un code GAUL (en France, un équivalent d'un département)
- ActionGeo_Lat, ActionGeo_Long : latitude et longitude, souvent un centroide de l'élément de localisation le plus précis (centroide de la région lorsque l'adm2 n'est pas renseigné par exemple)
- ActionGeo_FeatureID : un code GNIS lorsque la précision le permet, une chaine de caractère sinon (peut permettre des jointures spatiales)

En explorant rapidement les données, on s'aperçoit que la qualité des données est souvent assez variable et peu précise géographiquement. Par exemple, une donnée du 20 octobre 2026 est codée à Paris lorsque l'article commente des émeutes à Milan décrivant leur lien avec les émeutes en France. Pour s'affranchir de ce problème, une idée peut être de filtrer sur les évènements ayant un ActionGeo_Type de valeur 4 (et éventuellement aussi ceux de valeur 5).
On s'aperçoit aussi que des évènements encodés comme émeute concernent en réalité par exemple la mise en place d'un couvre feu par le premier ministre dans les grandes villes, ce qui n'est pas un évènement émeute au sens de notre analyse.

## Base de données proposée par GDELT Cloud
Face à la qualité de ces données, nous avons exploré une deuxième base de données proposée par GDELT Cloud, accessible via une API sous un "free" plan, qui semble proposer des données de bien meilleure qualité, tant spatialement que sur le lien réel de l'évènement avec une émeute, mais qui ne propose que des évènements se déroulant après mars 2026.

Cette API renvoie des données sous la forme d'un fichier JSON limité à 100 évènements sur une fenêtre temporelle d'au plus 30 jours. Pour explorer l'ensemble des évènements depuis mars 2026 (requête jusqu'au 10 novembre 2026), il faut donc parcourir successivement chacune des pages de 100 évènements des fenetres de 30 jours.

On obtient alors 82 évènements émeute, qui semblent être plutôt bien répartis sur le territoire français.





